# Single-cell RNA-seq Analysis (NACI2026)
This notebook performs standard scRNA-seq processing including QC, Normalization, Clustering, and Visualization for 3 patients (ZLF, ZFL, HJX).

In [ ]:
import scanpy as sc
import pandas as pd
import os
from scipy.io import mmread

# Set results directory
results_dir = './results'
if not os.path.exists(results_dir):
    os.makedirs(results_dir)
os.makedirs(os.path.join(results_dir, 'plots'), exist_ok=True)

sc.settings.figdir = os.path.join(results_dir, 'plots')

## 1. Load Data
We manually load the 10x matrix files because the features file contains a single column.

In [ ]:
def load_manual_10x(path):
    print(f"Loading from {path}...")
    mtx_path = os.path.join(path, 'matrix.mtx.gz')
    barcodes_path = os.path.join(path, 'barcodes.tsv.gz')
    features_path = os.path.join(path, 'features.tsv.gz')
    
    X = mmread(mtx_path).T.tocsr()
    barcodes = pd.read_csv(barcodes_path, header=None)[0].values
    features_df = pd.read_csv(features_path, header=None, sep='\t')
    
    gene_names = features_df[0].values
    adata = sc.AnnData(X=X)
    adata.obs_names = barcodes
    adata.var_names = gene_names
    return adata

data_base = './data/matrix'
samples = ['ZLF', 'ZFL', 'HJX']
adatas = []
for s in samples:
    adata = load_manual_10x(os.path.join(data_base, s))
    adata.obs['sample'] = s
    adata.var_names_make_unique()
    adatas.append(adata)

adata = sc.concat(adatas, index_unique='_')
print(f'Combined shape: {adata.shape}')

## 2. Quality Control

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-') | adata.var_names.str.startswith('mt-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], jitter=0.4, multi_panel=True)

## 3. Filtering

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[adata.obs.pct_counts_mt < 20, :]
adata = adata[adata.obs.total_counts < 40000, :]
print(f'Final shape: {adata.shape}')

## 4. Normalization and Dimension Reduction

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

adata_scaled = adata.copy()
sc.pp.scale(adata_scaled, max_value=10)
sc.tl.pca(adata_scaled, svd_solver='arpack')
sc.pp.neighbors(adata_scaled, n_neighbors=15, n_pcs=40)
sc.tl.umap(adata_scaled)

## 5. Clustering

In [ ]:
sc.tl.leiden(adata_scaled, resolution=0.5)
sc.pl.umap(adata_scaled, color=['leiden', 'sample'])

In [ ]:
# Export processed data (sparse to save space)
# adata_scaled.write(os.path.join(results_dir, 'processed_adata.h5ad'), compression='gzip')
print('Analysis and plotting complete.')